In [3]:
import os
import json
import random
import cv2
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score
)

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# ============================================================
# Reproducibility
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True


# ============================================================
# Paths and settings
# ============================================================

base = os.path.join(".", "post_disaster_split") 
img_dir = os.path.join(base, "images")
label_dir = os.path.join(base, "labels")

IMG_SIZE = 256
PATCH_SIZE = 128

NUM_CLASSES = 3
BATCH_SIZE = 8
EPOCHS = 30
LEARNING_RATE = 8e-4

PATCHES_PER_IMAGE_PER_EPOCH = 4

CAUTION_KERNEL_SIZE = 33

CLASS_NAMES = ["safe", "caution", "unsafe"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# ============================================================
# WKT polygon to OpenCV points
# ============================================================

def polygon_wkt_to_points(wkt):
    try:
        coord_text = wkt.split("((")[1].split("))")[0]
        points = []

        for pair in coord_text.split(","):
            parts = pair.strip().split()

            if len(parts) != 2:
                continue

            x = int(round(float(parts[0])))
            y = int(round(float(parts[1])))

            points.append([x, y])

        if len(points) < 3:
            return None

        return np.array(points, dtype=np.int32)

    except Exception:
        return None


# ============================================================
# JSON label to building mask
# ============================================================

def json_to_building_mask(json_path):
    with open(json_path, "r") as f:
        data = json.load(f)

    width = data.get("metadata", {}).get("width", 1024)
    height = data.get("metadata", {}).get("height", 1024)

    mask = np.zeros((height, width), dtype=np.uint8)

    features = data.get("features", {}).get("xy", [])

    for feature in features:
        props = feature.get("properties", {})

        if props.get("feature_type") != "building":
            continue

        wkt = feature.get("wkt", "")
        points = polygon_wkt_to_points(wkt)

        if points is not None:
            cv2.fillPoly(mask, [points], 1)

    return mask


# ============================================================
# Find matching label file
# ============================================================

def find_label_path(image_file):
    name_without_ext = os.path.splitext(image_file)[0]

    candidates = [
        os.path.join(label_dir, name_without_ext + ".json"),
        os.path.join(label_dir, name_without_ext + ".png"),
        os.path.join(label_dir, name_without_ext + "_target.png"),
    ]

    for path in candidates:
        if os.path.exists(path):
            return path

    return None


# ============================================================
# Load building mask
# ============================================================

def load_building_mask(label_path):
    if label_path.endswith(".json"):
        return json_to_building_mask(label_path)

    mask = cv2.imread(label_path, 0)

    if mask is None:
        return None

    return (mask > 0).astype(np.uint8)


# ============================================================
# UAV pseudo-label generator
# ============================================================

def create_uav_landing_mask(post, building_mask):
    """
    Classes:
        0 = safe
        1 = caution
        2 = unsafe
    """

    building_obstacle = (building_mask > 0).astype(np.uint8)

    post_gray = cv2.cvtColor(post, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(post, cv2.COLOR_BGR2HSV)

    s = hsv[:, :, 1]
    v = hsv[:, :, 2]

    b = post[:, :, 0].astype(np.int16)
    g = post[:, :, 1].astype(np.int16)
    r = post[:, :, 2].astype(np.int16)

    # -------------------------
    # Very dark terrain
    # -------------------------

    dark_terrain = (v < 45).astype(np.uint8)

    dark_terrain = cv2.morphologyEx(
        dark_terrain,
        cv2.MORPH_OPEN,
        np.ones((3, 3), np.uint8)
    )

    # -------------------------
    # Vegetation
    # -------------------------

    green_dominance = (
        (g > r + 15) &
        (g > b + 15) &
        (g > 75)
    ).astype(np.uint8)

    green_dominance = cv2.morphologyEx(
        green_dominance,
        cv2.MORPH_OPEN,
        np.ones((3, 3), np.uint8)
    )

    green_dominance = cv2.morphologyEx(
        green_dominance,
        cv2.MORPH_CLOSE,
        np.ones((5, 5), np.uint8)
    )

    # -------------------------
    # Cloud / smoke
    # -------------------------

    cloud_or_smoke = (
        (v > 180) &
        (s < 55)
    ).astype(np.uint8)

    cloud_or_smoke = cv2.morphologyEx(
        cloud_or_smoke,
        cv2.MORPH_OPEN,
        np.ones((3, 3), np.uint8)
    )

    # -------------------------
    # Texture variance
    # -------------------------

    gray_float = post_gray.astype(np.float32) / 255.0

    gray_mean = cv2.blur(gray_float, (7, 7))
    gray_sq_mean = cv2.blur(gray_float ** 2, (7, 7))
    texture_var = gray_sq_mean - (gray_mean ** 2)
    texture_var_norm = texture_var / (texture_var.max() + 1e-8)

    rough_threshold = max(0.07, np.percentile(texture_var_norm, 92))
    rough_terrain = (texture_var_norm > rough_threshold).astype(np.uint8)

    rough_terrain = cv2.morphologyEx(
        rough_terrain,
        cv2.MORPH_OPEN,
        np.ones((3, 3), np.uint8)
    )

    # -------------------------
    # Edge density
    # -------------------------

    edges = cv2.Canny(post, 80, 160)
    edges_binary = (edges > 0).astype(np.uint8)

    edge_density = cv2.blur(edges_binary.astype(np.float32), (9, 9))

    high_edge_area = (edge_density > 0.16).astype(np.uint8)
    medium_edge_area = (edge_density > 0.07).astype(np.uint8)

    # ==========================================
    # NEW: Water / Floodwater rules
    # ==========================================
    # Muddy water (Tan/Brown/Khaki hue + very smooth + no edges)
    muddy_water = (
        (hsv[:, :, 0] > 10) & (hsv[:, :, 0] < 45) &
        (texture_var_norm < 0.02) &
        (edge_density < 0.02)
    ).astype(np.uint8)

    # Clear/Deep water (Blue hue + very smooth)
    dark_water = (
        (hsv[:, :, 0] > 90) & (hsv[:, :, 0] < 140) &
        (texture_var_norm < 0.02)
    ).astype(np.uint8)

    water_mask = (muddy_water | dark_water).astype(np.uint8)
    
    # Clean up the water mask (removes tiny dots, fills small holes)
    water_mask = cv2.morphologyEx(water_mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    water_mask = cv2.morphologyEx(water_mask, cv2.MORPH_CLOSE, np.ones((9, 9), np.uint8))

    # -------------------------
    # Dense forest
    # -------------------------

    dense_forest = (
        (green_dominance == 1) &
        ((rough_terrain == 1) | (high_edge_area == 1))
    ).astype(np.uint8)

    dense_forest = cv2.morphologyEx(
        dense_forest,
        cv2.MORPH_CLOSE,
        np.ones((5, 5), np.uint8)
    )

    # -------------------------
    # Unsafe mask
    # -------------------------

    unsafe_mask = np.zeros_like(building_mask, dtype=np.uint8)

    unsafe_mask[building_obstacle == 1] = 1
    unsafe_mask[dark_terrain == 1] = 1
    unsafe_mask[cloud_or_smoke == 1] = 1
    unsafe_mask[dense_forest == 1] = 1
    unsafe_mask[water_mask == 1] = 1

    unsafe_mask = cv2.morphologyEx(
        unsafe_mask,
        cv2.MORPH_OPEN,
        np.ones((3, 3), np.uint8)
    )

    unsafe_mask = cv2.morphologyEx(
        unsafe_mask,
        cv2.MORPH_CLOSE,
        np.ones((5, 5), np.uint8)
    )

    # -------------------------
    # Caution mask
    # -------------------------

    kernel = np.ones((CAUTION_KERNEL_SIZE, CAUTION_KERNEL_SIZE), np.uint8)
    dilated_unsafe = cv2.dilate(unsafe_mask, kernel, iterations=1)

    landing_mask = np.zeros_like(building_mask, dtype=np.uint8)

    landing_mask[dilated_unsafe == 1] = 1
    landing_mask[green_dominance == 1] = 1
    landing_mask[rough_terrain == 1] = 1
    landing_mask[medium_edge_area == 1] = 1
    landing_mask[high_edge_area == 1] = 1

    landing_mask[unsafe_mask == 1] = 2

    return landing_mask


# ============================================================
# Feature generator
# ============================================================

def make_post_features(post):
    post_float = post.astype(np.float32) / 255.0

    post_gray = cv2.cvtColor(post, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
    post_gray_ch = post_gray[..., None]

    hsv = cv2.cvtColor(post, cv2.COLOR_BGR2HSV).astype(np.float32) / 255.0
    lab = cv2.cvtColor(post, cv2.COLOR_BGR2LAB).astype(np.float32) / 255.0

    edges = cv2.Canny(post, 80, 160).astype(np.float32) / 255.0
    edges_ch = edges[..., None]

    edge_density = cv2.blur(edges.astype(np.float32), (5, 5))
    edge_density_ch = edge_density[..., None]

    gray_mean = cv2.blur(post_gray, (5, 5))
    gray_sq_mean = cv2.blur(post_gray ** 2, (5, 5))
    texture_var = gray_sq_mean - (gray_mean ** 2)
    texture_var = texture_var / (texture_var.max() + 1e-8)
    texture_var_ch = texture_var[..., None]

    sobel_x = cv2.Sobel(post_gray, cv2.CV_32F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(post_gray, cv2.CV_32F, 0, 1, ksize=3)

    gradient_mag = np.sqrt(sobel_x ** 2 + sobel_y ** 2)
    gradient_mag = gradient_mag / (gradient_mag.max() + 1e-8)
    gradient_mag_ch = gradient_mag[..., None]

    features = np.concatenate(
        [
            post_float,
            post_gray_ch,
            hsv,
            lab,
            edges_ch,
            edge_density_ch,
            texture_var_ch,
            gradient_mag_ch
        ],
        axis=-1
    )

    return features.astype(np.float32)


# ============================================================
# Load data
# ============================================================

X = []
Y = []

print("Starting data preparation...")

files = sorted(os.listdir(img_dir))

for file in files:
    if not file.lower().endswith((".png", ".jpg", ".jpeg")):
        continue

    post_path = os.path.join(img_dir, file)
    label_path = find_label_path(file)

    if label_path is None:
        print("Missing label for:", file)
        continue

    post = cv2.imread(post_path)

    if post is None:
        print("Could not read image:", file)
        continue

    building_mask = load_building_mask(label_path)

    if building_mask is None:
        print("Could not read label:", label_path)
        continue

    post = cv2.resize(post, (IMG_SIZE, IMG_SIZE))

    building_mask = cv2.resize(
        building_mask,
        (IMG_SIZE, IMG_SIZE),
        interpolation=cv2.INTER_NEAREST
    )

    landing_mask = create_uav_landing_mask(post, building_mask)
    features = make_post_features(post)

    X.append(features)
    Y.append(landing_mask)

X = np.array(X, dtype=np.float32)
Y = np.array(Y, dtype=np.uint8)

if len(X) == 0:
    raise ValueError("No valid image/label pairs were loaded.")

print("Dataset shape:", X.shape, Y.shape)
print("Classes in masks:", np.unique(Y, return_counts=True))


# ============================================================
# Train / test split
# ============================================================

indices = np.arange(len(X))

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=SEED,
    shuffle=True
)

X_train = X[train_idx]
Y_train = Y[train_idx]

X_test = X[test_idx]
Y_test = Y[test_idx]


# ============================================================
# Class weights
# ============================================================

flat_y = Y_train.flatten()
counts = np.bincount(flat_y, minlength=NUM_CLASSES).astype(np.float32)

print("Training class counts:", counts)

freq = counts / counts.sum()

class_weights = 1.0 / np.sqrt(freq + 1e-8)
class_weights = class_weights / class_weights.mean()

# Updated:
# Slightly more unsafe pressure than the previous version.
# This aims to improve unsafe recall/F1 without over-predicting unsafe too much.
class_weights[0] *= 1.00
class_weights[1] *= 1.25
class_weights[2] *= 1.10

class_weights = class_weights / class_weights.mean()

class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

print("Class weights:", class_weights)


# ============================================================
# Balanced Patch Dataset
# ============================================================

class BalancedPatchDataset(Dataset):
    def __init__(
        self,
        images,
        masks,
        patch_size=128,
        samples_per_epoch=1000,
        min_fraction_by_class=None
    ):
        self.images = images
        self.masks = masks
        self.patch_size = patch_size
        self.samples_per_epoch = samples_per_epoch

        if min_fraction_by_class is None:
            min_fraction_by_class = {
                0: 0.20,
                1: 0.15,
                2: 0.12
            }

        self.min_fraction_by_class = min_fraction_by_class

        self.image_indices_by_class = {}

        for cls in range(NUM_CLASSES):
            valid_indices = []

            for i in range(len(self.masks)):
                if np.any(self.masks[i] == cls):
                    valid_indices.append(i)

            self.image_indices_by_class[cls] = valid_indices

        print("Images containing each class:")
        for cls in range(NUM_CLASSES):
            print(cls, len(self.image_indices_by_class[cls]))

    def __len__(self):
        return self.samples_per_epoch

    def random_crop_around_class(self, image, mask, cls):
        h, w = mask.shape
        ps = self.patch_size
        half = ps // 2

        coords = np.argwhere(mask == cls)

        if len(coords) == 0:
            cy = np.random.randint(0, h)
            cx = np.random.randint(0, w)
        else:
            cy, cx = coords[np.random.randint(len(coords))]

        image_pad = np.pad(
            image,
            ((half, half), (half, half), (0, 0)),
            mode="reflect"
        )

        mask_pad = np.pad(
            mask,
            ((half, half), (half, half)),
            mode="reflect"
        )

        cy += half
        cx += half

        y1 = cy - half
        y2 = y1 + ps

        x1 = cx - half
        x2 = x1 + ps

        image_crop = image_pad[y1:y2, x1:x2, :]
        mask_crop = mask_pad[y1:y2, x1:x2]

        return image_crop, mask_crop

    def get_good_crop(self, image, mask, cls, max_tries=20):
        min_fraction = self.min_fraction_by_class.get(cls, 0.05)

        best_image_crop = None
        best_mask_crop = None
        best_fraction = -1.0

        for _ in range(max_tries):
            image_crop, mask_crop = self.random_crop_around_class(image, mask, cls)
            fraction = np.mean(mask_crop == cls)

            if fraction > best_fraction:
                best_fraction = fraction
                best_image_crop = image_crop
                best_mask_crop = mask_crop

            if fraction >= min_fraction:
                return image_crop, mask_crop

        return best_image_crop, best_mask_crop

    def augment(self, image, mask):
        if np.random.rand() < 0.5:
            image = np.flip(image, axis=1).copy()
            mask = np.flip(mask, axis=1).copy()

        if np.random.rand() < 0.5:
            image = np.flip(image, axis=0).copy()
            mask = np.flip(mask, axis=0).copy()

        k = np.random.randint(0, 4)
        image = np.rot90(image, k, axes=(0, 1)).copy()
        mask = np.rot90(mask, k, axes=(0, 1)).copy()

        # brightness
        if np.random.rand() < 0.7:
            factor = np.random.uniform(0.85, 1.15)
            image = np.clip(image * factor, 0.0, 1.0)

        # slight noise
        if np.random.rand() < 0.25:
            noise = np.random.normal(0, 0.015, image.shape).astype(np.float32)
            image = np.clip(image + noise, 0.0, 1.0)

        return image, mask

    def __getitem__(self, idx):
        # Updated:
        # Slightly more unsafe patches than previous version.
        # Previous: [0.34, 0.44, 0.22]
        # New:      [0.32, 0.40, 0.28]
        target_class = np.random.choice(
            [0, 1, 2],
            p=[0.32, 0.40, 0.28]
        )

        valid_images = self.image_indices_by_class[target_class]

        if len(valid_images) == 0:
            img_idx = np.random.randint(0, len(self.images))
        else:
            img_idx = valid_images[np.random.randint(len(valid_images))]

        image = self.images[img_idx]
        mask = self.masks[img_idx]

        image_crop, mask_crop = self.get_good_crop(
            image,
            mask,
            target_class
        )

        image_crop, mask_crop = self.augment(image_crop, mask_crop)

        image_tensor = torch.tensor(image_crop, dtype=torch.float32).permute(2, 0, 1)
        mask_tensor = torch.tensor(mask_crop, dtype=torch.long)

        return image_tensor, mask_tensor


# ============================================================
# Full Image Dataset
# ============================================================

class FullImageDataset(Dataset):
    def __init__(self, images, masks):
        self.images = images
        self.masks = masks

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = torch.tensor(self.images[idx], dtype=torch.float32).permute(2, 0, 1)
        mask = torch.tensor(self.masks[idx], dtype=torch.long)

        return image, mask


samples_per_epoch = len(X_train) * PATCHES_PER_IMAGE_PER_EPOCH

train_dataset = BalancedPatchDataset(
    X_train,
    Y_train,
    patch_size=PATCH_SIZE,
    samples_per_epoch=samples_per_epoch
)

test_dataset = FullImageDataset(X_test, Y_test)

num_workers = 0

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=(device.type == "cuda")
)

test_loader = DataLoader(
    test_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=(device.type == "cuda")
)


# ============================================================
# Model: Residual U-Net Small
# ============================================================

class ResidualDoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=0.0):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(num_groups=8, num_channels=out_channels),
            nn.SiLU(inplace=True),

            nn.Dropout2d(dropout),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(num_groups=8, num_channels=out_channels),
        )

        if in_channels != out_channels:
            self.skip = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        else:
            self.skip = nn.Identity()

        self.act = nn.SiLU(inplace=True)

    def forward(self, x):
        return self.act(self.conv(x) + self.skip(x))


class UNetBetter(nn.Module):
    def __init__(self, in_channels=14, num_classes=3):
        super().__init__()

        self.enc1 = ResidualDoubleConv(in_channels, 32, dropout=0.02)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = ResidualDoubleConv(32, 64, dropout=0.03)
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = ResidualDoubleConv(64, 128, dropout=0.05)
        self.pool3 = nn.MaxPool2d(2)

        self.bottleneck = ResidualDoubleConv(128, 256, dropout=0.10)

        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3 = ResidualDoubleConv(256, 128, dropout=0.05)

        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = ResidualDoubleConv(128, 64, dropout=0.03)

        self.up1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec1 = ResidualDoubleConv(64, 32, dropout=0.02)

        self.final = nn.Conv2d(32, num_classes, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        p1 = self.pool1(e1)

        e2 = self.enc2(p1)
        p2 = self.pool2(e2)

        e3 = self.enc3(p2)
        p3 = self.pool3(e3)

        b = self.bottleneck(p3)

        u3 = self.up3(b)
        u3 = torch.cat([u3, e3], dim=1)
        d3 = self.dec3(u3)

        u2 = self.up2(d3)
        u2 = torch.cat([u2, e2], dim=1)
        d2 = self.dec2(u2)

        u1 = self.up1(d2)
        u1 = torch.cat([u1, e1], dim=1)
        d1 = self.dec1(u1)

        return self.final(d1)


model = UNetBetter(in_channels=14, num_classes=NUM_CLASSES).to(device)


# ============================================================
# Loss functions
# ============================================================

class SoftDiceLoss(nn.Module):
    def __init__(self, num_classes=3, smooth=1e-6):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=1)

        targets_one_hot = F.one_hot(targets, self.num_classes)
        targets_one_hot = targets_one_hot.permute(0, 3, 1, 2).float()

        dims = (0, 2, 3)

        intersection = torch.sum(probs * targets_one_hot, dims)
        cardinality = torch.sum(probs + targets_one_hot, dims)

        dice = (2.0 * intersection + self.smooth) / (cardinality + self.smooth)

        return 1.0 - dice.mean()


class TverskyLoss(nn.Module):
    def __init__(
        self,
        num_classes=3,
        alpha=None,
        beta=None,
        smooth=1e-6
    ):
        super().__init__()

        self.num_classes = num_classes
        self.smooth = smooth

        # Updated:
        # For unsafe class, beta is higher.
        # Higher beta means missing unsafe pixels is punished more.
        if alpha is None:
            alpha = torch.tensor([0.50, 0.60, 0.55], dtype=torch.float32)

        if beta is None:
            beta = torch.tensor([0.50, 0.50, 0.75], dtype=torch.float32)

        self.register_buffer("alpha", alpha)
        self.register_buffer("beta", beta)

    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=1)

        targets_one_hot = F.one_hot(targets, self.num_classes)
        targets_one_hot = targets_one_hot.permute(0, 3, 1, 2).float()

        dims = (0, 2, 3)

        true_positive = torch.sum(probs * targets_one_hot, dims)
        false_positive = torch.sum(probs * (1.0 - targets_one_hot), dims)
        false_negative = torch.sum((1.0 - probs) * targets_one_hot, dims)

        tversky = (
            true_positive + self.smooth
        ) / (
            true_positive
            + self.alpha * false_positive
            + self.beta * false_negative
            + self.smooth
        )

        return 1.0 - tversky.mean()


class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0):
        super().__init__()
        self.weight = weight
        self.gamma = gamma

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(
            logits,
            targets,
            weight=self.weight,
            reduction="none"
        )

        pt = torch.exp(-ce_loss)
        focal_loss = ((1.0 - pt) ** self.gamma) * ce_loss

        return focal_loss.mean()


class CombinedLoss(nn.Module):
    def __init__(self, class_weights):
        super().__init__()

        self.ce = nn.CrossEntropyLoss(weight=class_weights)
        self.dice = SoftDiceLoss(num_classes=NUM_CLASSES)
        self.tversky = TverskyLoss(num_classes=NUM_CLASSES)
        self.focal = FocalLoss(weight=class_weights, gamma=1.5)

    def forward(self, logits, targets):
        ce_loss = self.ce(logits, targets)
        dice_loss = self.dice(logits, targets)
        tversky_loss = self.tversky(logits, targets)
        focal_loss = self.focal(logits, targets)

        return (
            0.25 * ce_loss
            + 0.20 * dice_loss
            + 0.35 * tversky_loss
            + 0.20 * focal_loss
        )


criterion = CombinedLoss(class_weights).to(device)

optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=2e-4
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3
)

scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))


# ============================================================
# Training loop
# ============================================================

best_val_loss = float("inf")
best_model_path = "best_uav_unsafe_f1_unet.pth"

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for images, masks in train_loader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            outputs = model(images)
            loss = criterion(outputs, masks)

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

    avg_train_loss = running_loss / len(train_loader)

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for images, masks in test_loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, masks)

            val_loss += loss.item()

    avg_val_loss = val_loss / len(test_loader)

    scheduler.step(avg_val_loss)

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch [{epoch + 1}/{EPOCHS}] "
        f"Train Loss: {avg_train_loss:.4f} "
        f"Val Loss: {avg_val_loss:.4f} "
        f"LR: {current_lr:.6f}"
    )

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), best_model_path)
        print("Saved best model.")

print("Best validation loss:", best_val_loss)


# ============================================================
# Prediction helpers
# ============================================================

def predict_with_thresholds(probs, unsafe_threshold=0.55, caution_threshold=0.45):
    """
    probs shape:
        N, C, H, W

    Logic:
        1. Choose between safe/caution first.
        2. Predict caution if caution probability is high enough.
        3. Predict unsafe if unsafe probability is high enough.
    """

    caution_p = probs[:, 1]
    unsafe_p = probs[:, 2]

    safe_vs_caution = np.argmax(probs[:, 0:2], axis=1).astype(np.uint8)

    pred = safe_vs_caution.copy()

    pred[caution_p >= caution_threshold] = 1
    pred[unsafe_p >= unsafe_threshold] = 2

    return pred


def predict_with_thresholds_flat(probs_flat, unsafe_threshold=0.55, caution_threshold=0.45):
    """
    probs_flat shape:
        total_pixels, C
    """

    caution_p = probs_flat[:, 1]
    unsafe_p = probs_flat[:, 2]

    pred = np.argmax(probs_flat[:, 0:2], axis=1).astype(np.uint8)

    pred[caution_p >= caution_threshold] = 1
    pred[unsafe_p >= unsafe_threshold] = 2

    return pred


def evaluate_predictions(y_true_flat, y_pred_flat, title):
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)

    print("\nConfusion Matrix:")
    print(confusion_matrix(
        y_true_flat,
        y_pred_flat,
        labels=[0, 1, 2]
    ))

    print("\nClassification Report:")
    print(classification_report(
        y_true_flat,
        y_pred_flat,
        labels=[0, 1, 2],
        target_names=CLASS_NAMES,
        zero_division=0
    ))


def postprocess_unsafe_predictions(preds, min_area=25, close_kernel=3, dilate_kernel=3):
    """
    Cleans unsafe predictions:
        1. Closes small gaps inside unsafe regions.
        2. Removes tiny unsafe noise.
        3. Slightly expands unsafe into nearby caution pixels only.

    preds shape:
        N, H, W
    """

    processed = preds.copy()

    for i in range(processed.shape[0]):
        mask = processed[i].copy()

        unsafe = (mask == 2).astype(np.uint8)

        unsafe = cv2.morphologyEx(
            unsafe,
            cv2.MORPH_CLOSE,
            np.ones((close_kernel, close_kernel), np.uint8)
        )

        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
            unsafe,
            connectivity=8
        )

        cleaned = np.zeros_like(unsafe)

        for label_id in range(1, num_labels):
            area = stats[label_id, cv2.CC_STAT_AREA]

            if area >= min_area:
                cleaned[labels == label_id] = 1

        dilated = cv2.dilate(
            cleaned,
            np.ones((dilate_kernel, dilate_kernel), np.uint8),
            iterations=1
        )

        mask[cleaned == 1] = 2

        # Important:
        # expand unsafe only into caution, not directly into safe.
        mask[(dilated == 1) & (mask == 1)] = 2

        processed[i] = mask

    return processed


# ============================================================
# Load best model
# ============================================================

model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()


# ============================================================
# Collect probabilities on validation/test set
# ============================================================

all_probs = []
all_true = []

with torch.no_grad():
    for images, masks in test_loader:
        images = images.to(device, non_blocking=True)

        outputs = model(images)
        probs = F.softmax(outputs, dim=1).cpu().numpy()

        all_probs.append(probs)
        all_true.append(masks.numpy())

all_probs = np.concatenate(all_probs, axis=0)
all_true = np.concatenate(all_true, axis=0)

y_true_flat = all_true.flatten()


# ============================================================
# Evaluation 1: normal argmax
# ============================================================

argmax_preds = np.argmax(all_probs, axis=1).astype(np.uint8)
argmax_flat = argmax_preds.flatten()

evaluate_predictions(
    y_true_flat,
    argmax_flat,
    "Normal argmax evaluation"
)


# ============================================================
# Evaluation 2: post-processed argmax
# ============================================================

post_argmax_preds = postprocess_unsafe_predictions(
    argmax_preds,
    min_area=25,
    close_kernel=3,
    dilate_kernel=3
)

post_argmax_flat = post_argmax_preds.flatten()

evaluate_predictions(
    y_true_flat,
    post_argmax_flat,
    "Post-processed argmax evaluation"
)


# ============================================================
# Evaluation 3: threshold search
# ============================================================

best_score = -1e9
best_unsafe_threshold = 0.50
best_caution_threshold = 0.40
best_pred_flat = None

print("\nSearching best thresholds...")

# Flatten probabilities for faster threshold search
all_probs_flat = np.moveaxis(all_probs, 1, -1).reshape(-1, NUM_CLASSES)

# Speed-up:
# Search thresholds on a random subset of pixels.
# Final evaluation is still done on all pixels.
MAX_SEARCH_PIXELS = 2_000_000

if len(y_true_flat) > MAX_SEARCH_PIXELS:
    rng = np.random.default_rng(SEED)
    search_indices = rng.choice(
        len(y_true_flat),
        size=MAX_SEARCH_PIXELS,
        replace=False
    )

    search_probs_flat = all_probs_flat[search_indices]
    search_true_flat = y_true_flat[search_indices]
else:
    search_probs_flat = all_probs_flat
    search_true_flat = y_true_flat

for unsafe_threshold in np.arange(0.35, 0.76, 0.02):
    for caution_threshold in np.arange(0.35, 0.76, 0.02):

        threshold_flat_search = predict_with_thresholds_flat(
            search_probs_flat,
            unsafe_threshold=unsafe_threshold,
            caution_threshold=caution_threshold
        )

        macro_f1 = f1_score(
            search_true_flat,
            threshold_flat_search,
            labels=[0, 1, 2],
            average="macro",
            zero_division=0
        )

        unsafe_f1 = f1_score(
            search_true_flat,
            threshold_flat_search,
            labels=[2],
            average="macro",
            zero_division=0
        )

        unsafe_precision = precision_score(
            search_true_flat,
            threshold_flat_search,
            labels=[2],
            average="macro",
            zero_division=0
        )

        unsafe_recall = recall_score(
            search_true_flat,
            threshold_flat_search,
            labels=[2],
            average="macro",
            zero_division=0
        )

        caution_f1 = f1_score(
            search_true_flat,
            threshold_flat_search,
            labels=[1],
            average="macro",
            zero_division=0
        )

        # Updated:
        # This favours unsafe F1 and recall.
        # But it rejects thresholds where unsafe precision collapses too much.
        if unsafe_precision < 0.82:
            score = -1e9
        else:
            score = (
                0.30 * macro_f1
                + 0.40 * unsafe_f1
                + 0.20 * unsafe_recall
                + 0.10 * caution_f1
            )

        if score > best_score:
            best_score = score
            best_unsafe_threshold = float(unsafe_threshold)
            best_caution_threshold = float(caution_threshold)

print("\nBest threshold settings:")
print("Unsafe threshold:", best_unsafe_threshold)
print("Caution threshold:", best_caution_threshold)
print("Best threshold score:", best_score)

threshold_preds = predict_with_thresholds(
    all_probs,
    unsafe_threshold=best_unsafe_threshold,
    caution_threshold=best_caution_threshold
)

threshold_flat = threshold_preds.flatten()

evaluate_predictions(
    y_true_flat,
    threshold_flat,
    "Threshold-tuned evaluation"
)


# ============================================================
# Evaluation 4: threshold + post-processing
# ============================================================

post_threshold_preds = postprocess_unsafe_predictions(
    threshold_preds,
    min_area=25,
    close_kernel=3,
    dilate_kernel=3
)

post_threshold_flat = post_threshold_preds.flatten()

evaluate_predictions(
    y_true_flat,
    post_threshold_flat,
    "Threshold-tuned + post-processed evaluation"
)


# ============================================================
# Save threshold settings
# ============================================================

threshold_config = {
    "unsafe_threshold": best_unsafe_threshold,
    "caution_threshold": best_caution_threshold,
    "postprocess_min_area": 25,
    "postprocess_close_kernel": 3,
    "postprocess_dilate_kernel": 3
}

with open("best_thresholds_unsafe_f1.json", "w") as f:
    json.dump(threshold_config, f, indent=4)

print("\nSaved:")
print(best_model_path)
print("best_thresholds_unsafe_f1.json")

Using device: cuda
GPU: NVIDIA GeForce RTX 4070 Ti SUPER
Starting data preparation...
Dataset shape: (9168, 256, 256, 14) (9168, 256, 256)
Classes in masks: (array([0, 1, 2], dtype=uint8), array([ 60409286, 358776325, 181648437]))
Training class counts: [4.8480120e+07 2.8680006e+08 1.4536085e+08]
Class weights: tensor([1.3959, 0.7174, 0.8867], device='cuda:0')
Images containing each class:
0 6570
1 7333
2 7303


C:\Users\shash\AppData\Local\Temp\ipykernel_49768\2456442126.py:934: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
C:\Users\shash\AppData\Local\Temp\ipykernel_49768\2456442126.py:954: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):


Epoch [1/30] Train Loss: 0.3224 Val Loss: 0.2685 LR: 0.000800
Saved best model.
Epoch [2/30] Train Loss: 0.2349 Val Loss: 0.2153 LR: 0.000800
Saved best model.
Epoch [3/30] Train Loss: 0.2136 Val Loss: 0.2211 LR: 0.000800
Epoch [4/30] Train Loss: 0.2040 Val Loss: 0.2041 LR: 0.000800
Saved best model.
Epoch [5/30] Train Loss: 0.1948 Val Loss: 0.1859 LR: 0.000800
Saved best model.
Epoch [6/30] Train Loss: 0.1854 Val Loss: 0.1835 LR: 0.000800
Saved best model.
Epoch [7/30] Train Loss: 0.1788 Val Loss: 0.1762 LR: 0.000800
Saved best model.
Epoch [8/30] Train Loss: 0.1739 Val Loss: 0.1739 LR: 0.000800
Saved best model.
Epoch [9/30] Train Loss: 0.1702 Val Loss: 0.1751 LR: 0.000800
Epoch [10/30] Train Loss: 0.1686 Val Loss: 0.1670 LR: 0.000800
Saved best model.
Epoch [11/30] Train Loss: 0.1644 Val Loss: 0.1663 LR: 0.000800
Saved best model.
Epoch [12/30] Train Loss: 0.1628 Val Loss: 0.1712 LR: 0.000800
Epoch [13/30] Train Loss: 0.1600 Val Loss: 0.1661 LR: 0.000800
Saved best model.
Epoch [14/